# 1. 환경설정

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import koreanize_matplotlib
from sqlalchemy import create_engine

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

engine = create_engine(os.environ["DB_URL"])
conn = engine.connect()

In [ ]:
DATE_FMT = "%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f"

RETENTION_START_HOUR = 24
RETENTION_WINDOW_DAY = 7

In [ ]:
def _normalize_sql(sql: str) -> str:

    return sql.replace("%%", "%")


def run_query(query: str, name: str | None = None) -> pd.DataFrame:
    result = conn.exec_driver_sql(_normalize_sql(query))
    rows = result.fetchall()
    df = pd.DataFrame(rows, columns=result.keys())
    if name:
        print(f"[{name}] rows={len(df):,}, cols={len(df.columns):,}")
    return df


def execute_many(sql: str) -> None:
    statements = [stmt.strip() for stmt in sql.split(";") if stmt.strip()]
    for stmt in statements:
        conn.exec_driver_sql(_normalize_sql(stmt))
    try:
        conn.commit()
    except Exception:
        pass
    print(f"Executed {len(statements):,} statements.")

# 2. 데이터 전처리

### VIEW 생성 및 결측값 확인

- 이 과정에서 user_id 결측 + event_time 변환 실패 행 제거

In [ ]:
create_views_sql = """
-- 1) v_events_signup
DROP VIEW IF EXISTS v_events_signup;
CREATE VIEW v_events_signup AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time
FROM events_signup
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

-- 2) v_events_content_start
DROP VIEW IF EXISTS v_events_content_start;
CREATE VIEW v_events_content_start AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id
FROM events_content_start
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

-- 3) v_events_lesson_view
DROP VIEW IF EXISTS v_events_lesson_view;
CREATE VIEW v_events_lesson_view AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id,
    `lesson_id`  AS lesson_id
FROM events_lesson_view
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

-- 4) v_events_lesson_complete
DROP VIEW IF EXISTS v_events_lesson_complete;
CREATE VIEW v_events_lesson_complete AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id,
    `lesson_id`  AS lesson_id
FROM events_lesson_complete
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

-- 5) v_events_content_end
DROP VIEW IF EXISTS v_events_content_end;
CREATE VIEW v_events_content_end AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id` AS content_id
FROM events_content_end
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;

-- 6) v_events_related_question_click
DROP VIEW IF EXISTS v_events_related_question_click;
CREATE VIEW v_events_related_question_click AS
SELECT
    user_id,
    STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') AS event_time,
    `content_id`  AS content_id,
    `lesson_id`   AS lesson_id
FROM events_related_question_click
WHERE user_id IS NOT NULL
  AND user_id <> ''
  AND STR_TO_DATE(event_ts, '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f') IS NOT NULL;
"""

execute_many(create_views_sql)

### VIEW 생성 여부 검증

In [ ]:
run_query("""
SELECT 'v_events_signup'        AS view_name, COUNT(*) AS row_cnt FROM v_events_signup
UNION ALL SELECT 'v_events_content_start',           COUNT(*) FROM v_events_content_start
UNION ALL SELECT 'v_events_lesson_view',       COUNT(*) FROM v_events_lesson_view
UNION ALL SELECT 'v_events_lesson_complete',         COUNT(*) FROM v_events_lesson_complete
UNION ALL SELECT 'v_events_content_end',             COUNT(*) FROM v_events_content_end
UNION ALL SELECT 'v_events_related_question_click',  COUNT(*) FROM v_events_related_question_click;
""", "view_check")

### 결측값 확인

In [ ]:
DATE_FMT = '%%%%Y-%%%%m-%%%%d %%%%H:%%%%i:%%%%s.%%%%f'

null_check_df = run_query(f"""
SELECT 'events_signup' AS table_name,
    COUNT(*) AS total,
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END) AS user_id_null,
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END) AS event_time_null
FROM events_signup
UNION ALL
SELECT 'events_content_start', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_content_start
UNION ALL
SELECT 'events_lesson_view', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_lesson_view
UNION ALL
SELECT 'events_lesson_complete', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_lesson_complete
UNION ALL
SELECT 'events_content_end', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_content_end
UNION ALL
SELECT 'events_related_question_click', COUNT(*),
    SUM(CASE WHEN NULLIF(TRIM(user_id), '') IS NULL THEN 1 ELSE 0 END),
    SUM(CASE WHEN STR_TO_DATE(event_ts, '{DATE_FMT}') IS NULL THEN 1 ELSE 0 END)
FROM events_related_question_click;
""", "null_check")

null_check_df

In [ ]:
duplicate_check_df = run_query("""
SELECT 'events_signup' AS table_name,
    COUNT(*) AS total,
    COUNT(DISTINCT user_id, event_time) AS unique_cnt,
    COUNT(*) - COUNT(DISTINCT user_id, event_time) AS duplicated_cnt
FROM v_events_signup
UNION ALL
SELECT 'events_content_start', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_content_start
UNION ALL
SELECT 'events_lesson_view', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_lesson_view
UNION ALL
SELECT 'events_lesson_complete', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_lesson_complete
UNION ALL
SELECT 'events_content_end', COUNT(*), COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_content_end
UNION ALL
SELECT 'events_related_question_click', COUNT(*),
    COUNT(DISTINCT user_id, event_time),
    COUNT(*) - COUNT(DISTINCT user_id, event_time)
FROM v_events_related_question_click;
""", "duplicate_check")

duplicate_check_df

### 이상치 1

In [ ]:
time_range_df = run_query("""
SELECT 'v_events_signup' AS view_name,
    MIN(event_time) AS min_t, MAX(event_time) AS max_t,
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END) AS future_cnt
FROM v_events_signup
UNION ALL
SELECT 'v_events_content_start', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_content_start
UNION ALL
SELECT 'v_events_lesson_view', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_lesson_view
UNION ALL
SELECT 'v_events_lesson_complete', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_lesson_complete
UNION ALL
SELECT 'v_events_content_end', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_content_end
UNION ALL
SELECT 'v_events_related_question_click', MIN(event_time), MAX(event_time),
    SUM(CASE WHEN event_time > NOW() THEN 1 ELSE 0 END)
FROM v_events_related_question_click;
""", "time_range")

time_range_df

### 이상치 2 : 가입 전 활동 (정합성)

In [ ]:
before_signup_df = run_query("""
WITH signup AS (
    SELECT user_id, MIN(event_time) AS signup_time
    FROM v_events_signup GROUP BY user_id
)
SELECT 'v_events_content_start' AS view_name,
    COUNT(*) AS before_signup_rows
FROM v_events_content_start sc
JOIN signup s ON sc.user_id = s.user_id
WHERE sc.event_time < s.signup_time
UNION ALL
SELECT 'v_events_lesson_view', COUNT(*)
FROM v_events_lesson_view l
JOIN signup s ON l.user_id = s.user_id
WHERE l.event_time < s.signup_time
UNION ALL
SELECT 'v_events_lesson_complete', COUNT(*)
FROM v_events_lesson_complete cl
JOIN signup s ON cl.user_id = s.user_id
WHERE cl.event_time < s.signup_time
UNION ALL
SELECT 'v_events_content_end', COUNT(*)
FROM v_events_content_end ec
JOIN signup s ON ec.user_id = s.user_id
WHERE ec.event_time < s.signup_time
UNION ALL
SELECT 'v_events_related_question_click', COUNT(*)
FROM v_events_related_question_click cq
JOIN signup s ON cq.user_id = s.user_id
WHERE cq.event_time < s.signup_time;
""", "before_signup")

before_signup_df

### 이상치 3 : 봇 의심 (한 유저가 너무 많은 이벤트)

- 유저별 일평균 활동량 분포 살펴보기

In [ ]:
user_stats_query = """
    SELECT
        user_id,
        COUNT(*) AS event_cnt,
        TIMESTAMPDIFF(DAY, MIN(event_time), MAX(event_time)) AS active_days,
        COUNT(*) / GREATEST(TIMESTAMPDIFF(DAY, MIN(event_time), MAX(event_time)), 1) AS daily_avg
    FROM v_events_lesson_view
    GROUP BY user_id
"""
df_user_stats = run_query(user_stats_query)

In [ ]:
# 기본 통계
print("전체 유저 수:", len(df_user_stats))
print("\n=== daily_avg 분포 ===")
print(df_user_stats['daily_avg'].describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]))

# 임계값별 잘리는 유저 수 시뮬레이션
print("\n=== 임계값별 봇으로 분류되는 유저 수 ===")
for threshold in [20, 30, 50, 100, 200, 500, 974]:
    bot_count = (df_user_stats['daily_avg'] >= threshold).sum()
    pct = bot_count / len(df_user_stats) * 100
    print(f"  ≥ {threshold:>4}/일 : {bot_count:>6}명 ({pct:.3f}%)")

- 전체 11.4만 명

- 974/일 이상: 8명
- 500/일 이상: 93명
- 200/일 이상: 441명

In [ ]:
# 500/일 이상 유저들의 분포 추가 확인
top_suspects = df_user_stats[df_user_stats['daily_avg'] >= 500].sort_values('daily_avg', ascending=False)
print(f"500/일 이상 유저: {len(top_suspects)}명")
print("\n=== 상위 20명 ===")
print(top_suspects[['user_id', 'event_cnt', 'active_days', 'daily_avg']].head(20))

print("\n=== 일평균 분포 (500+) ===")
print(top_suspects['daily_avg'].describe())

- 20명 중 1명만 빼고 active_days가 0~1으로 확인됨
- 하루(또는 몇 시간) 동안 1,000번 이상 레슨에 진입했다

In [ ]:
# active_days = 0 인 유저들의 실제 데이터 확인
sample_user = 'SAMPLE_USER_ID'

run_query(f"""
SELECT
    user_id,
    event_time,
    lesson_id,
    COUNT(*) AS dup_cnt
FROM v_events_lesson_view
WHERE user_id = '{sample_user}'
GROUP BY user_id, event_time, lesson_id
ORDER BY dup_cnt DESC
LIMIT 5;
""", "active0_check")

In [ ]:
# pandas로 봇 후보 추출 (즉시)
THRESHOLD = 200
bot_candidates = df_user_stats[df_user_stats['daily_avg'] >= THRESHOLD].copy()
print(f"봇 후보: {len(bot_candidates)}명")

# bot_users 테이블 생성
execute_many("""
DROP TABLE IF EXISTS bot_users;
CREATE TABLE bot_users (
    user_id     VARCHAR(64) NOT NULL PRIMARY KEY,
    event_cnt   INT,
    active_days INT,
    daily_avg   DECIMAL(10,2),
    detected_at DATETIME DEFAULT CURRENT_TIMESTAMP
) ENGINE=InnoDB;
""")

# pandas DataFrame을 한 번에 INSERT
bot_candidates[['user_id', 'event_cnt', 'active_days', 'daily_avg']].to_sql(
    'bot_users', con=engine, if_exists='append', index=False
)

# 확인
run_query("SELECT COUNT(*) AS cnt FROM bot_users", "bot_count")

# 3. Activation

## 1. 유저 정의

In [ ]:
query = """
WITH signup AS (
    SELECT
        user_id,
        MIN(event_time) AS signup_time
    FROM v_events_signup
    GROUP BY user_id
),

content AS (
    SELECT
        s.user_id,
        MIN(c.event_time) AS first_content_time
    FROM signup s
    JOIN v_events_content_start c
        ON s.user_id = c.user_id
       AND c.event_time >= s.signup_time
    GROUP BY s.user_id
),

lesson AS (
    SELECT
        c.user_id,
        MIN(l.event_time) AS first_lesson_time
    FROM content c
    JOIN v_events_lesson_view l
        ON c.user_id = l.user_id
       AND l.event_time >= c.first_content_time
    GROUP BY c.user_id
),

complete AS (
    SELECT
        l.user_id,
        MIN(cp.event_time) AS first_complete_time
    FROM lesson l
    JOIN v_events_lesson_complete cp
        ON l.user_id = cp.user_id
       AND cp.event_time >= l.first_lesson_time
    GROUP BY l.user_id
)

SELECT
    COUNT(*) AS activation_users
FROM complete;
"""

activation_df = pd.read_sql(query, conn)
activation_df

In [ ]:
query = """
WITH signup AS (
    SELECT
        s.user_id,
        MIN(s.event_time) AS signup_time
    FROM v_events_signup s
    LEFT JOIN bot_users b
        ON s.user_id = b.user_id
    WHERE b.user_id IS NULL
    GROUP BY s.user_id
),

content AS (
    SELECT
        s.user_id,
        MIN(c.event_time) AS first_content_time
    FROM signup s
    JOIN v_events_content_start c
        ON s.user_id = c.user_id
       AND c.event_time >= s.signup_time
    GROUP BY s.user_id
),

lesson AS (
    SELECT
        c.user_id,
        MIN(l.event_time) AS first_lesson_time
    FROM content c
    JOIN v_events_lesson_view l
        ON c.user_id = l.user_id
       AND l.event_time >= c.first_content_time
    GROUP BY c.user_id
),

complete_users AS (
    SELECT
        l.user_id,
        MIN(cp.event_time) AS first_complete_time
    FROM lesson l
    JOIN v_events_lesson_complete cp
        ON l.user_id = cp.user_id
       AND cp.event_time >= l.first_lesson_time
    GROUP BY l.user_id
)

SELECT
    COUNT(*) AS activation_users
FROM complete_users;
"""

activation_df = pd.read_sql(query, conn)
activation_df

In [ ]:
query = """
WITH signup AS (
    SELECT
        s.user_id,
        MIN(s.event_time) AS signup_time
    FROM v_events_signup s
    LEFT JOIN bot_users b
        ON s.user_id = b.user_id
    WHERE b.user_id IS NULL
    GROUP BY s.user_id
),

first_content AS (
    SELECT
        sc.user_id,
        MIN(sc.event_time) AS first_content_time
    FROM v_events_content_start sc
    JOIN signup s
        ON sc.user_id = s.user_id
       AND sc.event_time >= s.signup_time
    GROUP BY sc.user_id
),

first_lesson AS (
    SELECT
        el.user_id,
        MIN(el.event_time) AS first_lesson_time
    FROM v_events_lesson_view el
    JOIN first_content fc
        ON el.user_id = fc.user_id
       AND el.event_time >= fc.first_content_time
    GROUP BY el.user_id
),

activation_users AS (
    SELECT
        cl.user_id,
        MIN(cl.event_time) AS activation_time
    FROM v_events_lesson_complete cl
    JOIN first_lesson fl
        ON cl.user_id = fl.user_id
       AND cl.event_time >= fl.first_lesson_time
    GROUP BY cl.user_id
)

SELECT
    COUNT(DISTINCT user_id) AS activation_users
FROM activation_users;
"""

activation_df = pd.read_sql(query, engine)
activation_df

## 1.2 유저 시각화

In [ ]:
query = """
WITH signup AS (
    SELECT
        s.user_id,
        MIN(s.event_time) AS signup_time
    FROM v_events_signup s
    LEFT JOIN bot_users b
        ON s.user_id = b.user_id
    WHERE b.user_id IS NULL
    GROUP BY s.user_id
),

content AS (
    SELECT
        s.user_id,
        MIN(c.event_time) AS first_content_time
    FROM signup s
    JOIN v_events_content_start c
        ON s.user_id = c.user_id
       AND c.event_time >= s.signup_time
    GROUP BY s.user_id
),

lesson AS (
    SELECT
        c.user_id,
        MIN(l.event_time) AS first_lesson_time
    FROM content c
    JOIN v_events_lesson_view l
        ON c.user_id = l.user_id
       AND l.event_time >= c.first_content_time
    GROUP BY c.user_id
),

activation_users AS (
    SELECT
        l.user_id,
        MIN(cp.event_time) AS activation_time
    FROM lesson l
    JOIN v_events_lesson_complete cp
        ON l.user_id = cp.user_id
       AND cp.event_time >= l.first_lesson_time
    GROUP BY l.user_id
),

funnel_counts AS (
    SELECT '1. 회원가입' AS step, COUNT(*) AS users, 1 AS step_order
    FROM signup

    UNION ALL

    SELECT '2. 콘텐츠 시작' AS step, COUNT(*) AS users, 2 AS step_order
    FROM content

    UNION ALL

    SELECT '3. 레슨 진입' AS step, COUNT(*) AS users, 3 AS step_order
    FROM lesson

    UNION ALL

    SELECT '4. 레슨 완료(Activation)' AS step, COUNT(*) AS users, 4 AS step_order
    FROM activation_users
)

SELECT
    step_order,
    step,
    users,
    ROUND(users * 100.0 / FIRST_VALUE(users) OVER (ORDER BY step_order), 1) AS convert_from_signup_pct,
    ROUND(users * 100.0 / LAG(users) OVER (ORDER BY step_order), 1) AS step_convert_pct
FROM funnel_counts
ORDER BY step_order;
"""

ac_funnel_df = pd.read_sql(query, engine)
ac_funnel_df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plot_df = ac_funnel_df.copy()

steps = plot_df['step'].tolist()
counts = pd.to_numeric(plot_df['users'], errors='coerce').fillna(0)

rates = plot_df['step_convert_pct'].tolist()

plt.figure(figsize=(10, 6))

colors = ['#bdd8f1', '#82a6cb', '#3667a6', '#214177']
plt.bar(steps, counts, color=colors)

max_count = counts.max()

for i, count in enumerate(counts):
    plt.text(
        i,
        count + (max_count * 0.02),
        f'{int(count):,}명',
        ha='center',
        va='bottom',
        fontsize=12,
        fontweight='bold'
    )

    if i > 0 and pd.notna(rates[i]):
        plt.text(
            i,
            count / 2,
            f'({rates[i]}%)',
            ha='center',
            va='center',
            color='white',
            fontsize=11,
            fontweight='bold'
        )

ax = plt.gca()
ax.spines[['top', 'left', 'right']].set_visible(False)
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))

plt.title('활성화 퍼널', fontsize=15, fontweight='bold')
plt.ylabel('User Count')
plt.ylim(0, max_count * 1.15)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 2. 온보딩 속도

In [ ]:
query = '''
WITH signup AS (
    SELECT
        s.user_id,
        MIN(s.event_time) AS signup_time
    FROM v_events_signup s
    LEFT JOIN bot_users b
        ON s.user_id = b.user_id
    WHERE b.user_id IS NULL
    GROUP BY s.user_id
),

content AS (
    SELECT
        s.user_id,
        MIN(c.event_time) AS first_content_time
    FROM signup s
    JOIN v_events_content_start c
        ON s.user_id = c.user_id
       AND c.event_time >= s.signup_time
    GROUP BY s.user_id
),

lesson AS (
    SELECT
        c.user_id,
        MIN(l.event_time) AS first_lesson_time
    FROM content c
    JOIN v_events_lesson_view l
        ON c.user_id = l.user_id
       AND l.event_time >= c.first_content_time
    GROUP BY c.user_id
),

complete AS (
    SELECT
        l.user_id,
        MIN(cp.event_time) AS first_complete_time
    FROM lesson l
    JOIN v_events_lesson_complete cp
        ON l.user_id = cp.user_id
       AND cp.event_time >= l.first_lesson_time
    GROUP BY l.user_id
),

user_diff AS (
    SELECT
        s.user_id,

        DATEDIFF(c.first_content_time, s.signup_time) AS diff_signup_to_content,
        DATEDIFF(l.first_lesson_time, c.first_content_time) AS diff_content_to_lesson,
        DATEDIFF(cp.first_complete_time, l.first_lesson_time) AS diff_lesson_to_complete,
        DATEDIFF(cp.first_complete_time, s.signup_time) AS diff_total_funnel

    FROM signup s
    LEFT JOIN content c ON s.user_id = c.user_id
    LEFT JOIN lesson l ON c.user_id = l.user_id
    LEFT JOIN complete cp ON l.user_id = cp.user_id
)

SELECT
    COUNT(*) AS total_signup_users,

    ROUND(AVG(diff_signup_to_content), 1) AS avg_days_to_content,

    ROUND(SUM(CASE WHEN diff_signup_to_content = 0 THEN 1 ELSE 0 END) * 100.0
        / NULLIF(COUNT(diff_signup_to_content), 0), 2) AS day0_content_pct,

    ROUND(AVG(diff_content_to_lesson), 1) AS avg_days_to_lesson,

    ROUND(SUM(CASE WHEN diff_content_to_lesson = 0 THEN 1 ELSE 0 END) * 100.0
        / NULLIF(COUNT(diff_content_to_lesson), 0), 2) AS day0_lesson_pct,

    ROUND(AVG(diff_lesson_to_complete), 1) AS avg_days_to_complete,

    ROUND(SUM(CASE WHEN diff_lesson_to_complete = 0 THEN 1 ELSE 0 END) * 100.0
        / NULLIF(COUNT(diff_lesson_to_complete), 0), 2) AS day0_complete_pct,

    ROUND(AVG(diff_total_funnel), 1) AS avg_days_total

FROM user_diff
'''

summary_funnel_df = pd.read_sql(query, engine)
summary_funnel_df

## 2.2 온보딩 속도 시각화

In [ ]:
raw = '''
WITH signup AS (
    SELECT
        s.user_id,
        MIN(s.event_time) AS signup_time
    FROM v_events_signup s
    LEFT JOIN bot_users b
        ON s.user_id = b.user_id
    WHERE b.user_id IS NULL
    GROUP BY s.user_id
),

content AS (
    SELECT
        s.user_id,
        MIN(c.event_time) AS first_content_time
    FROM signup s
    JOIN v_events_content_start c
        ON s.user_id = c.user_id
       AND c.event_time >= s.signup_time
    GROUP BY s.user_id
),

lesson AS (
    SELECT
        c.user_id,
        MIN(l.event_time) AS first_lesson_time
    FROM content c
    JOIN v_events_lesson_view l
        ON c.user_id = l.user_id
       AND l.event_time >= c.first_content_time
    GROUP BY c.user_id
),

complete AS (
    SELECT
        l.user_id,
        MIN(cp.event_time) AS first_complete_time
    FROM lesson l
    JOIN v_events_lesson_complete cp
        ON l.user_id = cp.user_id
       AND cp.event_time >= l.first_lesson_time
    GROUP BY l.user_id
),

user_diff AS (
    SELECT
        s.user_id,

        DATEDIFF(
            c.first_content_time,
            s.signup_time
        ) AS diff_signup_to_content,

        DATEDIFF(
            l.first_lesson_time,
            c.first_content_time
        ) AS diff_content_to_lesson,

        DATEDIFF(
            cp.first_complete_time,
            l.first_lesson_time
        ) AS diff_lesson_to_complete,

        DATEDIFF(
            cp.first_complete_time,
            s.signup_time
        ) AS diff_total_funnel

    FROM signup s

    LEFT JOIN content c
        ON s.user_id = c.user_id

    LEFT JOIN lesson l
        ON c.user_id = l.user_id

    LEFT JOIN complete cp
        ON l.user_id = cp.user_id
)

SELECT
    diff_signup_to_content,
    diff_content_to_lesson,
    diff_lesson_to_complete,
    diff_total_funnel
FROM user_diff
'''

funnel_diff_df = pd.read_sql(raw, engine)

In [ ]:
bins = [-1, 0, 1, 3, 7, 14, 30, float('inf')]
labels = ['당일', '1일', '2-3일', '4-7일', '8-14일', '15-30일', '30일 초과']

cols = [
    'diff_signup_to_content',
    'diff_content_to_lesson',
    'diff_lesson_to_complete',
    'diff_total_funnel'
]

titles = [
    '회원가입 → 콘텐츠 시작',
    '콘텐츠 시작 → 레슨 시작',
    '레슨 시작 → 레슨 완강',
    '회원가입 → 레슨 완강'
]

colors = ['#bdd8f1', '#82a6cb', '#3667a6', '#214177']

for idx, (col, title) in enumerate(zip(cols, titles)):
    plot_df = funnel_diff_df[funnel_diff_df[col].notna()].copy()

    plot_df['period'] = pd.cut(
        plot_df[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    dist_df = (
        plot_df['period']
        .value_counts()
        .reindex(labels, fill_value=0)
        .reset_index()
    )

    dist_df.columns = ['period', 'users']

    total_users = dist_df['users'].sum()
    dist_df['rate'] = (dist_df['users'] / total_users * 100).round(1)

    plt.figure(figsize=(9, 5))

    plt.bar(
        dist_df['period'],
        dist_df['users'],
        color=colors[idx]
    )

    max_users = dist_df['users'].max()

    for i, row in dist_df.iterrows():
        if row['users'] > 0:
            plt.text(
                i,
                row['users'] + max_users * 0.02,
                f"{int(row['users']):,}명\n({row['rate']}%)",
                ha='center',
                va='bottom',
                fontsize=10
            )

    ax = plt.gca()
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))

    plt.title(f'{title} 소요일 분포', fontsize=14, fontweight='bold')
    plt.xlabel('소요일 구간')
    plt.ylabel('User Count')
    plt.ylim(0, max_users * 1.2)
    plt.grid(axis='y', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.show()

## 3. TVT

In [ ]:
query = '''
WITH signup AS (
    SELECT
        s.user_id,
        MIN(s.event_time) AS signup_time
    FROM v_events_signup s
    LEFT JOIN bot_users b
        ON s.user_id = b.user_id
    WHERE b.user_id IS NULL
    GROUP BY s.user_id
),

content AS (
    SELECT
        s.user_id,
        MIN(c.event_time) AS first_content_time
    FROM signup s
    JOIN v_events_content_start c
        ON s.user_id = c.user_id
       AND c.event_time >= s.signup_time
    GROUP BY s.user_id
),

lesson AS (
    SELECT
        c.user_id,
        MIN(l.event_time) AS first_lesson_time
    FROM content c
    JOIN v_events_lesson_view l
        ON c.user_id = l.user_id
       AND l.event_time >= c.first_content_time
    GROUP BY c.user_id
),

complete AS (
    SELECT
        l.user_id,
        MIN(cp.event_time) AS first_complete_time
    FROM lesson l
    JOIN v_events_lesson_complete cp
        ON l.user_id = cp.user_id
       AND cp.event_time >= l.first_lesson_time
    GROUP BY l.user_id
),

user_timeline AS (
    SELECT
        s.user_id,
        s.signup_time,
        c.first_content_time,
        l.first_lesson_time,
        cp.first_complete_time

    FROM signup s

    LEFT JOIN content c
        ON s.user_id = c.user_id

    LEFT JOIN lesson l
        ON c.user_id = l.user_id

    LEFT JOIN complete cp
        ON l.user_id = cp.user_id
)

SELECT
    user_id,

    ROUND(
        TIMESTAMPDIFF(
            MINUTE,
            signup_time,
            first_content_time
        ) / 60.0,
        2
    ) AS TVT_signup_to_content,

    ROUND(
        TIMESTAMPDIFF(
            MINUTE,
            first_content_time,
            first_lesson_time
        ) / 60.0,
        2
    ) AS TVT_content_to_lesson,

    ROUND(
        TIMESTAMPDIFF(
            MINUTE,
            first_lesson_time,
            first_complete_time
        ) / 60.0,
        2
    ) AS TVT_lesson_to_complete

FROM user_timeline
'''

tvt_df = pd.read_sql(query, conn)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

bins = [0, 1, 3, 6, 12, 24, 48, float('inf')]

labels = [
    '1시간 이내',
    '1-3시간',
    '3-6시간',
    '6-12시간',
    '12-24시간',
    '1-2일',
    '2일 초과'
]

tvt_cols = [
    'TVT_signup_to_content',
    'TVT_content_to_lesson',
    'TVT_lesson_to_complete'
]

titles = [
    '1. 가입 → 콘텐츠 시작',
    '2. 콘텐츠 시작 → 레슨 진입',
    '3. 레슨 진입 → 레슨 완강'
]

colors = ['#bdd8f1', '#82a6cb', '#3667a6', '#214177', '#bdd8f1', '#82a6cb', '#3667a6', '#999999']

fig, ax = plt.subplots(1, 3, figsize=(18, 6))

for i, col in enumerate(tvt_cols):

    if i == 0:
        step_df = tvt_df.copy()
    else:
        step_df = tvt_df[tvt_df[tvt_cols[i - 1]].notnull()].copy()

    step_df[col + '_cat'] = pd.cut(
        step_df[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    step_df[col + '_cat'] = (
        step_df[col + '_cat']
        .cat.add_categories('이탈 유저')
        .fillna('이탈 유저')
    )

    summary = (
        step_df[col + '_cat']
        .value_counts()
        .reindex(labels + ['이탈 유저'], fill_value=0)
    )

    ax[i].bar(
        summary.index.astype(str),
        summary.values,
        color=colors
    )

    max_value = summary.max()

    for j, v in enumerate(summary.values):
        ax[i].text(
            j,
            v + (max_value * 0.02),
            f'{v:,.0f}',
            ha='center',
            va='bottom',
            fontweight='bold'
        )

    ax[i].spines[['top', 'left', 'right']].set_visible(False)
    ax[i].yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))
    ax[i].grid(axis='y', linestyle='--', alpha=0.3)

    ax[i].set_title(titles[i], fontsize=15, fontweight='bold')
    ax[i].set_xlabel('소요 시간')
    ax[i].set_ylabel('유저 수(명)')
    ax[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. 콘텐츠 난이도별 TVT

In [ ]:
query = """
WITH signup AS (
    SELECT
        s.user_id,
        MIN(s.event_time) AS signup_time
    FROM v_events_signup s
    LEFT JOIN bot_users b
        ON s.user_id = b.user_id
    WHERE NULLIF(s.user_id, '') IS NOT NULL
      AND b.user_id IS NULL
    GROUP BY s.user_id
),

first_content AS (
    SELECT
        user_id,
        content_id,
        event_time AS first_content_time
    FROM (
        SELECT
            s.user_id,
            sc.content_id,
            sc.event_time,
            ROW_NUMBER() OVER (
                PARTITION BY s.user_id
                ORDER BY sc.event_time, sc.content_id
            ) AS rn
        FROM signup s
        JOIN v_events_content_start sc
            ON s.user_id = sc.user_id
           AND sc.event_time >= s.signup_time
        WHERE NULLIF(sc.user_id, '') IS NOT NULL
    ) t
    WHERE rn = 1
),

first_lesson AS (
    SELECT
        fc.user_id,
        MIN(l.event_time) AS first_lesson_time
    FROM first_content fc
    JOIN v_events_lesson_view l
        ON fc.user_id = l.user_id
       AND l.event_time >= fc.first_content_time
    WHERE NULLIF(l.user_id, '') IS NOT NULL
    GROUP BY fc.user_id
),

first_complete AS (
    SELECT
        fl.user_id,
        MIN(cp.event_time) AS first_complete_time
    FROM first_lesson fl
    JOIN v_events_lesson_complete cp
        ON fl.user_id = cp.user_id
       AND cp.event_time >= fl.first_lesson_time
    WHERE NULLIF(cp.user_id, '') IS NOT NULL
    GROUP BY fl.user_id
)

SELECT
    fc.user_id,
    cm.difficulty,

    ROUND(
        TIMESTAMPDIFF(MINUTE, s.signup_time, fc.first_content_time) / 60.0,
        2
    ) AS TVT_signup_to_content,

    ROUND(
        TIMESTAMPDIFF(MINUTE, fc.first_content_time, fl.first_lesson_time) / 60.0,
        2
    ) AS TVT_content_to_lesson,

    ROUND(
        TIMESTAMPDIFF(MINUTE, fl.first_lesson_time, fcp.first_complete_time) / 60.0,
        2
    ) AS TVT_lesson_to_complete

FROM first_content fc
JOIN signup s
    ON fc.user_id = s.user_id

LEFT JOIN first_lesson fl
    ON fc.user_id = fl.user_id

LEFT JOIN first_complete fcp
    ON fc.user_id = fcp.user_id

LEFT JOIN content_dim cm
    ON fc.content_id = cm.content_id
"""

tvt_diff_df = pd.read_sql(query, conn)

tvt_diff_df.head()

In [ ]:
difficulty_tvt_summary = (
    tvt_diff_df
    .groupby('difficulty')[
        [
            'TVT_signup_to_content',
            'TVT_content_to_lesson',
            'TVT_lesson_to_complete'
        ]
    ]
    .mean()
    .round(1)
)

difficulty_tvt_summary

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

bins = [0, 1, 3, 6, 12, 24, 48, float('inf')]

labels = [
    '1시간 이내',
    '1-3시간',
    '3-6시간',
    '6-12시간',
    '12-24시간',
    '1-2일',
    '2일 초과'
]

tvt_cols = [
    'TVT_signup_to_content',

    'TVT_content_to_lesson',
    'TVT_lesson_to_complete'
]

titles = [
    '1. 가입 → 콘텐츠 시작',
    '2. 콘텐츠 시작 → 레슨 진입',
    '3. 레슨 진입 → 레슨 완강'
]

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

for i, col in enumerate(tvt_cols):

    if i == 0:
        step_df = tvt_diff_df.copy()

    else:
        step_df = tvt_diff_df[
            tvt_diff_df[tvt_cols[i - 1]].notnull()
        ].copy()

    step_df[col + '_cat'] = pd.cut(
        step_df[col],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    step_df[col + '_cat'] = (
        step_df[col + '_cat']
        .cat.add_categories('이탈 유저')
        .fillna('이탈 유저')
    )

    crosstab = pd.crosstab(
        step_df['difficulty'],
        step_df[col + '_cat']
    )

    crosstab = crosstab.reindex(
        columns=labels + ['이탈 유저'],
        fill_value=0
    )

    order = [
        'beginner',
        'intermediate',
        'advanced',
        'hard'
    ]

    valid_order = [
        x for x in order
        if x in crosstab.index
    ]

    if valid_order:
        crosstab = crosstab.loc[valid_order]

    crosstab_pct = (
        crosstab.div(
            crosstab.sum(axis=1),
            axis=0
        ) * 100
    )

    crosstab_pct.plot(
        kind='bar',
        stacked=True,
        ax=axes[i],
        colormap='PuBu',
        edgecolor='white',
        width=0.7,
        linewidth=0.5
    )

    axes[i].set_title(
        titles[i],
        fontsize=15,
        fontweight='bold',
        pad=15
    )

    axes[i].set_xlabel(
        '콘텐츠 난이도',
        fontweight='bold'
    )

    axes[i].set_ylabel(
        '유저 비율 (%)',
        fontweight='bold'
    )

    axes[i].tick_params(axis='x', rotation=0)

    axes[i].set_ylim(0, 100)

    axes[i].yaxis.set_major_formatter(
        mtick.PercentFormatter()
    )

    axes[i].spines[['top', 'right']].set_visible(False)

    if i == 2:

        axes[i].legend(
            title='소요 시간',
            bbox_to_anchor=(1.05, 1),
            loc='upper left'
        )

    else:

        legend = axes[i].get_legend()

        if legend is not None:
            legend.remove()

plt.tight_layout()

plt.show()